In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "litsearch"


In [2]:
from src.ie.extract_from_candidates import run_ie_on_candidates
from pathlib import Path
DATA_DIR = Path("..") / "data" / "processed" / "litsearch"


In [3]:
import re, json
from collections import Counter

TOKEN_RE = re.compile(r"\w+", flags=re.UNICODE)
STOP_WORDS = {
    "the","and","for","with","that","this","are","from","using","use",
    "a","an","of","in","to","on","by","is","as","we","be","which"
}

def simple_keyphrases(text: str, topk: int = 5):
    if not text:
        return []
    tokens = [t.lower() for t in TOKEN_RE.findall(text) if len(t) > 2]
    tokens = [t for t in tokens if t not in STOP_WORDS]
    common = [w for w,_ in Counter(tokens).most_common(topk)]
    return common

def simple_summary(text: str, max_sents: int = 2):
    if not text:
        return ""
    # naive sentence split
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return " ".join(sents[:max_sents]).strip()

def simple_llm_emulator(prompt: str) -> str:
    """
    Из prompt вытащим abstract (по маркерам), но для простоты можно
    ожидать, что prompt содержит 'Paper abstract:' и далее текст.
    Возвращаем JSON-строку, как это делал бы LLM.
    """
    m = re.search(r"Paper abstract:\n(.*?)(?:\n\n|$)", prompt, flags=re.S)
    abstract = m.group(1).strip() if m else ""
    kps = simple_keyphrases(abstract, topk=5)
    summ = simple_summary(abstract, max_sents=2)
    out = {"keyphrases": kps, "summary": summ}
    return json.dumps(out, ensure_ascii=False)


In [4]:
def dummy_llm(prompt: str) -> str:
    return """
{
  "keyphrases": ["information retrieval", "ranking", "language models"],
  "summary": "This paper studies a retrieval and ranking approach using language models."
}
"""


In [5]:
run_ie_on_candidates(
    candidates_path=DATA_DIR / "candidates.jsonl",
    corpus_path=DATA_DIR / "corpus.jsonl",
    output_path=DATA_DIR / "ie_documents.jsonl",
    llm_call_fn=simple_llm_emulator,
)


In [6]:
!head -n 5 ../data/processed/litsearch/ie_documents.jsonl | jq .

{
  "doc_id": 221995575,
  "title": "Contrastive Distillation on Intermediate Representations for Language Model Compression",
  "keyphrases": [
    "knowledge",
    "intermediate",
    "representations",
    "large",
    "layers"
  ],
  "summary": "Existing language model compression methods mostly use a simple L 2 loss to distill knowledge in the intermediate representations of a large BERT model to a smaller one. Although widely used, this objective by design assumes that all the dimensions of hidden representations are independent, failing to capture important structural knowledge in the intermediate layers of the teacher network.",
  "query_id": 0
}
{
  "doc_id": 259137871,
  "title": "GKD: A General Knowledge Distillation Framework for Large-scale Pre-trained Language Model",
  "keyphrases": [
    "distillation",
    "scale",
    "methods",
    "plms",
    "knowledge"
  ],
  "summary": "Currently, the reduction in the parameter scale of large-scale pre-trained language models (PL

In [7]:
import json


empty = 0
total = 0
path = DATA_DIR / "ie_documents.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        total += 1
        if not json.loads(line).get("summary"):
            empty += 1

print(f"Empty summaries: {empty}/{total}")


Empty summaries: 0/119400


In [8]:
from collections import defaultdict

by_query = defaultdict(list)

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        by_query[item["query_id"]].append(item)

qid = list(by_query.keys())[0]

for doc in by_query[qid][:5]:
    print(doc["title"])
    print(doc["keyphrases"])
    print(doc["summary"])
    print("-" * 40)


Contrastive Distillation on Intermediate Representations for Language Model Compression
['knowledge', 'intermediate', 'representations', 'large', 'layers']
Existing language model compression methods mostly use a simple L 2 loss to distill knowledge in the intermediate representations of a large BERT model to a smaller one. Although widely used, this objective by design assumes that all the dimensions of hidden representations are independent, failing to capture important structural knowledge in the intermediate layers of the teacher network.
----------------------------------------
GKD: A General Knowledge Distillation Framework for Large-scale Pre-trained Language Model
['distillation', 'scale', 'methods', 'plms', 'knowledge']
Currently, the reduction in the parameter scale of large-scale pre-trained language models (PLMs) through knowledge distillation has greatly facilitated their widespread deployment on various devices. However, the deployment of knowledge distillation systems fa

In [11]:
!head -n 3 ../data/processed/litsearch/candidates.jsonl | jq .

{
  "query_id": 0,
  "query": "Are there any research papers on methods to compress large-scale language models using task-agnostic knowledge distillation techniques?",
  "candidates": [
    {
      "doc_id": 221995575,
      "score": 37.03681734794637
    },
    {
      "doc_id": 259137871,
      "score": 35.94452799168773
    },
    {
      "doc_id": 256900817,
      "score": 35.15916143210283
    },
    {
      "doc_id": 215238664,
      "score": 33.40335362994966
    },
    {
      "doc_id": 218502458,
      "score": 33.185204149058876
    },
    {
      "doc_id": 257038997,
      "score": 33.1211812165141
    },
    {
      "doc_id": 201670719,
      "score": 32.84098755482068
    },
    {
      "doc_id": 222290473,
      "score": 32.83436270137386
    },
    {
      "doc_id": 248227350,
      "score": 31.559868374984262
    },
    {
      "doc_id": 226226888,
      "score": 31.40150515415201
    },
    {
      "doc_id": 233024733,
      "score": 31.166919505455738
    },
    {
  

In [14]:
# сколько строк (сколько запросов)
!wc -l ../data/processed/litsearch/candidates.jsonl

597 ../data/processed/litsearch/candidates.jsonl


In [15]:
# сколько уникальных query_id и какие они
!jq -r '.query_id' ../data/processed/litsearch/candidates.jsonl | sort -n | uniq -c | head -n 20

      1 0
      1 1
      1 2
      1 3
      1 4
      1 5
      1 6
      1 7
      1 8
      1 9
      1 10
      1 11
      1 12
      1 13
      1 14
      1 15
      1 16
      1 17
      1 18
      1 19


In [17]:
!jq -r '.query' ../data/processed/litsearch/candidates.jsonl | sed 's/^\s*//' | head -n 5


Are there any research papers on methods to compress large-scale language models using task-agnostic knowledge distillation techniques?
Are there any resources available for translating Tunisian Arabic dialect that contain both manually translated comments by native speakers and additional data augmented through methods like segmentation at stop words level?
Are there any studies that explore post-hoc techniques for hallucination detection at both the token- and sentence-level in neural sequence generation tasks?
Are there any tools or studies that have focused on building a morphological analyzer specifically for handling multiple Arabic dialects?
Are there papers that propose contextualized calibration for the probability of answers in language models?
sed: couldn't write 192 items to stdout: Broken pipe
Error: writing output failed: Broken pipe


In [19]:
!jq -r '.query_id' ../data/processed/litsearch/ie_documents.jsonl | sort -n | uniq -c | head -n 10

    200 0
    200 1
    200 2
    200 3
    200 4
    200 5
    200 6
    200 7
    200 8
    200 9
uniq: write error: Broken pipe


In [20]:
!sed -n '190,210p' ../data/processed/litsearch/ie_documents.jsonl | jq '.query_id'

0
0
0
0
0
0
0
0
0
0
0
1
1
1
1
1
1
1
1
1
1
